# Implicit differentiation

Gradients of *converged* quantities: `implicit_density` exposes the SCF fixed
point as a differentiable function of the energy functional (CPHF as a
`custom_vjp`), so a polarizability is one `jax.jacobian` through the converged
density — no field stepping.

In [1]:
import jax; jax.config.update("jax_enable_x64", True)
import equinox as eqx
import jax.numpy as jnp
import numpy as np
from dftax import KS, Molecule, becke, implicit_density, polarizability
from dftax.integrals.multipole import dipole_matrices
from dftax.ks.properties import nuclear_dipole
from dftax.energy.xc import PBE

mol = Molecule.from_xyz("O 0 0 0; H 0.7586 0 0.5043; H 0.7586 0 -0.5043", "sto-3g")
ks0 = KS(mol, PBE(), grid=becke(50, 110))
D = dipole_matrices(ks0.basis)
nuc = nuclear_dipole(mol)

def mu(field):
    ksf = eqx.tree_at(lambda k: k.hcore, ks0, ks0.hcore + jnp.einsum("i,ipq->pq", field, D))
    return nuc - jnp.einsum("ipq,pq->i", D, implicit_density(ksf))

alpha = jax.jacobian(mu)(jnp.zeros(3))
alpha = 0.5 * (alpha + alpha.T)
print("analytic polarizability (a.u.):\n", np.asarray(alpha))

analytic polarizability (a.u.):
 [[ 3.63668612e+00  1.39507933e-16  2.99760217e-15]
 [ 1.39507933e-16  1.24669654e-02 -2.44913410e-16]
 [ 2.99760217e-15 -2.44913410e-16  2.74638085e+00]]


In [2]:
alpha_fd = np.asarray(polarizability(mol, PBE(), method="fd", grid=becke(50, 110)))
print("finite-field reference:\n", alpha_fd)
print("max |diff| =", float(np.abs(np.asarray(alpha) - alpha_fd).max()))

finite-field reference:
 [[ 3.63668680e+00 -1.73523455e-12  2.08166817e-13]
 [-1.73523455e-12  1.24675044e-02  9.75348978e-14]
 [ 2.08166817e-13  9.75348978e-14  2.74636579e+00]]
max |diff| = 1.5059176729614876e-05
